# TrashTrack — Run B: Merged Training (Kaggle)
**MSDS 498 Capstone · Group 4**

Train on TACO + UAVVaste + Glasgow (merged) and evaluate per-dataset.

**Before running:** Settings (right sidebar) → Accelerator → **GPU P100**

**Add dataset:** Add Input → search `trashtrack-datasets` → Add


## 1. Setup

In [ ]:
# Verify GPU
import torch
assert torch.cuda.is_available(), "GPU not enabled! Go to Settings → Accelerator → GPU P100"
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Clone repo
!git clone https://github.com/Nandini1Bag/trashtrack.git /kaggle/working/trashtrack
%cd /kaggle/working/trashtrack

In [ ]:
# Install dependencies
!pip install -q ultralytics

## 2. Load Datasets

In [ ]:
# Find and unzip the dataset
import glob

# Kaggle mounts inputs at /kaggle/input/<dataset-name>/
zip_candidates = glob.glob("/kaggle/input/**/datasets.zip", recursive=True)
if zip_candidates:
    zip_path = zip_candidates[0]
    print(f"Found: {zip_path}")
else:
    # Maybe it was unzipped already or has a different structure
    print("Listing /kaggle/input/:")
    !ls -la /kaggle/input/
    !ls -la /kaggle/input/*/ 2>/dev/null
    raise FileNotFoundError("datasets.zip not found — did you Add Input?")

!unzip -qo {zip_path} -d /kaggle/working/trashtrack/
!mkdir -p datasets
!mv -f taco uavvaste rolid11k merged datasets/ 2>/dev/null
print("Datasets ready!")

## 3. Verify Datasets

In [ ]:
import os

def count_files(path, ext):
    if not os.path.exists(path):
        return 0
    return len([f for f in os.listdir(path) if f.lower().endswith(ext)])

print("Dataset Summary:")
print("=" * 55)
for ds in ["taco", "uavvaste", "rolid11k", "merged"]:
    for split in ["train", "val"]:
        imgs = count_files(f"datasets/{ds}/images/{split}", ".jpg") + \
               count_files(f"datasets/{ds}/images/{split}", ".png")
        lbls = count_files(f"datasets/{ds}/labels/{split}", ".txt")
        if imgs > 0:
            print(f"  {ds:12s} {split:5s}  {imgs:6d} images  {lbls:6d} labels")
print("=" * 55)
print("All good — ready to train.")

## 4. Run B — Merged Training
⏱ ~2-2.5 hours on P100.

In [ ]:
!yolo detect train \
    data=configs/merged.yaml \
    model=yolov8s.pt \
    epochs=50 \
    imgsz=640 \
    batch=16 \
    patience=15 \
    name=run_b_merged \
    save=True \
    plots=True

## 5. Evaluate on Each Dataset (ST-04)

In [ ]:
for ds, config, target in [
    ("TACO", "configs/taco.yaml", 0.45),
    ("UAVVaste", "configs/uavvaste.yaml", 0.35),
    ("Glasgow", "configs/rolid11k.yaml", 0.30),
]:
    print(f"\n{'='*50}")
    print(f"  {ds}  (target mAP@0.5 ≥ {target})")
    print(f"{'='*50}")

    !yolo detect val \
        data={config} \
        model=runs/detect/run_b_merged/weights/best.pt \
        name=run_b_eval_{ds.lower()} \
        plots=True

## 6. Results Summary

In [ ]:
import pandas as pd
from pathlib import Path

def read_results(run_dir):
    csv_path = Path(run_dir) / "results.csv"
    if not csv_path.exists():
        return None
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()
    return df.iloc[-1] if len(df) > 0 else None

r = read_results("runs/detect/run_b_merged")
if r is not None:
    print("📊 Run B (Merged) — Training Results")
    print("=" * 50)
    print(f"  mAP@0.5      : {float(r.get('metrics/mAP50(B)', 0)):.4f}")
    print(f"  mAP@0.5:0.95 : {float(r.get('metrics/mAP50-95(B)', 0)):.4f}")
    print(f"  Precision     : {float(r.get('metrics/precision(B)', 0)):.4f}")
    print(f"  Recall        : {float(r.get('metrics/recall(B)', 0)):.4f}")

print("\n\nRun A (TACO only, from Colab):")
print("  mAP@0.5 = 0.492 | P = 0.666 | R = 0.453")

print("\n\n🎯 Success Thresholds:")
print("─" * 40)
for ds, t in [("TACO", 0.45), ("UAVVaste", 0.35), ("Glasgow", 0.30)]:
    print(f"  {ds:12s}  mAP@0.5 ≥ {t}")

## 7. Save Weights

In [ ]:
# Copy best weights to /kaggle/working/ so they appear in the Output tab
import shutil
src = "runs/detect/run_b_merged/weights/best.pt"
dst = "/kaggle/working/run_b_best.pt"
shutil.copy2(src, dst)
print(f"Saved: {dst}")
print("\nDownload from the Output tab after this notebook finishes.")
print("Then on your local machine:")
print("  TT_MODEL_WEIGHTS=weights/best.pt uvicorn trashtrack.api:app --port 8000")